In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
import umap

np.random.seed(42)

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)
plt.scatter(X[:, 0], X[:, 1], alpha=0.6)
plt.title('Synthetic Blob Data (unlabelled)')
plt.show()

In [ ]:
def kmeans_from_scratch(X, k, n_iterations=100, seed=42):
    rng = np.random.RandomState(seed)
    # Initialize centroids by picking k random points
    initial_idx = rng.choice(len(X), k, replace=False)
    centroids = X[initial_idx].copy()

    for _ in range(n_iterations):
        # Assign each point to nearest centroid
        distances = np.sqrt(((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2).sum(axis=2))
        labels = np.argmin(distances, axis=1)

        # Update centroids
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])

        # Check for convergence
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    return labels, centroids


my_labels, my_centroids = kmeans_from_scratch(X, k=4, seed=42)

plt.scatter(X[:, 0], X[:, 1], c=my_labels, cmap='viridis', alpha=0.6)
plt.scatter(my_centroids[:, 0], my_centroids[:, 1], c='red', marker='X', s=200)
plt.title('My k-Means From Scratch')
plt.show()

In [ ]:
sklearn_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
sklearn_labels = sklearn_kmeans.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X[:, 0], X[:, 1], c=my_labels, cmap='viridis', alpha=0.6)
axes[0].set_title('My k-Means')
axes[1].scatter(X[:, 0], X[:, 1], c=sklearn_labels, cmap='viridis', alpha=0.6)
axes[1].set_title('sklearn KMeans')
plt.show()

print("Cluster assignments should look the same (labels may be permuted differently)")

In [ ]:
k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(k_values, inertias, marker='o')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_values, silhouette_scores, marker='o')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(silhouette_scores)]
print(f"Best k by silhouette score: {best_k}")

In [ ]:
final_km = KMeans(n_clusters=4, random_state=42, n_init=10)
final_labels = final_km.fit_predict(X)

plt.scatter(X[:, 0], X[:, 1], c=final_labels, cmap='viridis', alpha=0.6)
plt.scatter(final_km.cluster_centers_[:, 0], final_km.cluster_centers_[:, 1], 
            c='red', marker='X', s=200)
plt.title('Final Clustering (k=4)')
plt.show()

for i in range(4):
    cluster_points = X[final_labels == i]
    print(f"Cluster {i}: center=({cluster_points[:,0].mean():.2f}, {cluster_points[:,1].mean():.2f}), size={len(cluster_points)}")

## Choosing k and Naming Clusters

Both the elbow method (where inertia's rate of decrease slows 
down) and the silhouette score (which peaks at k=4) agree that 
**k=4** is the right choice for this dataset.

Naming the four clusters based on their position and spread:
- **Cluster 0**: "Top-left group" — compact, distinct from others
- **Cluster 1**: "Top-right group" — moderately spread
- **Cluster 2**: "Bottom-left group" — tight cluster
- **Cluster 3**: "Bottom-right group" — largest spread

(In a real business context, these would be named based on what 
the underlying data represents — e.g., customer segments like 
"high-value loyal", "price-sensitive", etc.)

In [ ]:
datasets = {
    'Blobs': make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)[0],
    'Moons': make_moons(n_samples=300, noise=0.08, random_state=42)[0],
    'Circles': make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)[0]
}

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row, (name, data) in enumerate(datasets.items()):
    data_scaled = StandardScaler().fit_transform(data)
    
    km_labels = KMeans(n_clusters=2 if name != 'Blobs' else 4, random_state=42, n_init=10).fit_predict(data_scaled)
    dbscan_labels = DBSCAN(eps=0.3, min_samples=5).fit_predict(data_scaled)
    agg_labels = AgglomerativeClustering(n_clusters=2 if name != 'Blobs' else 4).fit_predict(data_scaled)
    
    axes[row, 0].scatter(data[:, 0], data[:, 1], c=km_labels, cmap='viridis', alpha=0.6)
    axes[row, 0].set_title(f'{name}: k-Means')
    
    axes[row, 1].scatter(data[:, 0], data[:, 1], c=dbscan_labels, cmap='viridis', alpha=0.6)
    axes[row, 1].set_title(f'{name}: DBSCAN')
    
    axes[row, 2].scatter(data[:, 0], data[:, 1], c=agg_labels, cmap='viridis', alpha=0.6)
    axes[row, 2].set_title(f'{name}: Agglomerative')

plt.tight_layout()
plt.show()

## Clustering Algorithm Comparison

- **Blobs**: All three algorithms perform well since clusters are 
  roughly spherical and well-separated — this is the "easy case" 
  that k-means is designed for.

- **Moons**: k-means fails here because it assumes spherical 
  clusters, but the moon shapes are curved — it splits them 
  incorrectly. DBSCAN succeeds because it groups based on density 
  connectivity, correctly following the curved shape. Agglomerative 
  clustering (with default linkage) also struggles similarly to 
  k-means.

- **Circles**: Similarly, k-means and agglomerative clustering fail 
  on concentric circles since they can't separate based on radius 
  alone using centroid distance. DBSCAN handles this well because 
  it identifies dense regions regardless of shape.

**Key takeaway**: k-means assumes convex, similarly-sized clusters. 
DBSCAN handles arbitrary shapes and noise but requires tuning eps 
and min_samples. The right algorithm depends entirely on the 
underlying cluster shape.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Original dimensions: {X_digits.shape[1]}")

pca = PCA()
pca.fit(X_digits)

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Explained Variance (Digits Dataset)')
plt.legend()
plt.show()

n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Components needed for 95% variance: {n_components_95}")

In [ ]:
pca_2d = PCA(n_components=2).fit_transform(X_digits)
tsne_2d = TSNE(n_components=2, random_state=42).fit_transform(X_digits)
umap_2d = umap.UMAP(n_components=2, random_state=42).fit_transform(X_digits)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

scatter1 = axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=y_digits, cmap='tab10', alpha=0.6, s=10)
axes[0].set_title('PCA')

scatter2 = axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=y_digits, cmap='tab10', alpha=0.6, s=10)
axes[1].set_title('t-SNE')

scatter3 = axes[2].scatter(umap_2d[:, 0], umap_2d[:, 1], c=y_digits, cmap='tab10', alpha=0.6, s=10)
axes[2].set_title('UMAP')

plt.colorbar(scatter3, ax=axes[2], label='Digit')
plt.tight_layout()
plt.show()

## PCA vs t-SNE vs UMAP

**PCA** preserves global linear structure and variance, but the 
digit classes overlap significantly in just 2 dimensions since PCA 
only captures linear relationships — it's best for understanding 
overall variance, not for visual cluster separation.

**t-SNE** creates much tighter, more visually separated clusters 
because it focuses on preserving local neighbourhood structure, 
making it excellent for visualisation — but distances between 
clusters aren't meaningful, and it's not deterministic in the same 
way (though we fixed the seed here).

**UMAP** achieves similar (often better) separation to t-SNE but 
runs faster and preserves more of the global structure between 
clusters, making relative cluster positions somewhat more 
meaningful than t-SNE.

**Important**: PCA can be used for actual dimensionality reduction 
before modelling. t-SNE and UMAP are for visualisation only — they 
shouldn't be used as preprocessing for downstream models since 
their transformations aren't easily invertible or stable for new data.

In [ ]:
iso_forest = IsolationForest(contamination=0.05, random_state=42)
anomaly_labels = iso_forest.fit_predict(X)  # -1 = anomaly, 1 = normal

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=anomaly_labels, cmap='coolwarm', alpha=0.6)
plt.title('Isolation Forest Anomaly Detection (red = anomaly)')
plt.show()

print(f"Number of anomalies detected: {(anomaly_labels == -1).sum()}")